## Read in packages

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import polars.selectors as cs
import polars as pl
_=pl.Config.set_tbl_cols(100000)
_=pl.Config.set_tbl_rows(10000)
_=pl.Config.set_tbl_width_chars(10000)
_=pl.Config.set_fmt_str_lengths(10000)


In [ ]:
rbp = 
cell_line =

In [6]:
hash_table_path = "/project/PlatigLab/users/yogi/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/06_first_order_SHAP_analysis/outputs/hash_metadata/hash_metadata.tsv"

# K562

## Load in BAT for K562 and rMATS data for all RBPs of interest

In [7]:
# Load in the BAT for K562
# Function to get path for big table

from pathlib import Path

def get_path(cell_line: str, path_file: str = "data_path.txt") -> str:
    """
    Reads a base directory path from a text file and returns the full path
    to the cell line data directory.

    Args:
        cell_line (str): The name of the cell line (e.g. "K562" or "HepG2").
        path_file (str): Path to the text file containing the base directory path.

    Returns:
        str: Full path to the data file for the given cell line.
    """
    # Read base path from file
    base_path = Path(path_file).read_text().strip()
    
    # Build full path
    full_path = Path(base_path) / f"{cell_line}_all-data.feather"
    
    return str(full_path)

In [8]:
# BAT K562
data_path_K562 = get_path("K562")
BAT_K562 = pl.read_ipc(data_path_K562)
print(BAT_K562.shape)

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


(6291182, 1700)


In [9]:
# Make a list of RBPs with Crispr KD
rbps = ["IGF2BP1"]

## Main function to get all matched events (locations) between Crispr KD and the BAT

In [10]:
# Add description here

def load_rmats_K562(rbp_list, BAT):

    for rbp in rbp_list:

    # Load in rMATS data, drop unnecessary columns

        # Read in rMATS
        rMATS_data = pl.read_csv(f'/project/PlatigLab/data/ENCORE2026/rMATS_analysis/v29_rMATS_RBPKD/{rbp}-CRISPR-K562/SE.MATS.JC.txt', separator='\t')

        # Drop columns
        rMATS_filtered = rMATS_data[["FDR", "geneSymbol", "strand", "chr", "exonStart_0base", "exonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE", "ID", "IncFormLen", "SkipFormLen", "PValue", "IncLevelDifference"]]
        
        # Print shape of rMATS
        print(f"The shape of the rMATS file for {rbp} in K562 is {rMATS_filtered.shape}")

        # Grab all three exon combos from rMATS data and makes a 2 col. df with "ID" and "string"
        rMATS_events = rMATS_filtered.select([
            pl.col("ID"),
            pl.when(pl.col("strand") == "+")
            .then(
                pl.concat_str([
                    pl.col("chr"),
                    pl.col("strand"),
                    pl.col("upstreamES"),
                    pl.col("upstreamEE"),
                    pl.col("exonStart_0base"),
                    pl.col("exonEnd"),
                    pl.col("downstreamES"),
                    pl.col("downstreamEE")
                ], separator="_")
            )
            .otherwise(
                pl.concat_str([
                    pl.col("chr"),
                    pl.col("strand"),
                    pl.col("downstreamEE"),
                    pl.col("downstreamES"),
                    pl.col("exonEnd"),
                    pl.col("exonStart_0base"),
                    pl.col("upstreamEE"),
                    pl.col("upstreamES")
                ], separator="_")
            )
            .alias("string")
        ])

        # Define the number of elements to keep
        N_ELEMENTS = 8
        # Define the delimiter for splitting and joining
        DELIMITER = "_"
        
        BAT_K562_extracted = BAT.with_columns(
            pl.col("index")
            .str.split(by=DELIMITER)
            .list.slice(0, N_ELEMENTS)  # Slice the first N elements of the resulting list
            .list.join(DELIMITER)      # Join the list back into a single string using the delimiter
            .alias("string")
        )
        
        # Join on the string column; filtered
        # This makes the BAT with the matched events
        RBP_K562_KD_matches_BAT = BAT_K562_extracted.join(
            rMATS_events.select("string"),
            on="string",
             how="inner"
        ).drop("string")
        
        # Only need binding pattern -> drop all columns with "_shap"
        RBP_K562_KD_matches_BAT = RBP_K562_KD_matches_BAT.drop(pl.col("^.*_shap.*$"))

        print(f"Here is the shape of matches between {rbp} KD events and the BAT for K562:")
        print(RBP_K562_KD_matches_BAT.shape)

        # Get just control rows
        CTRL_rows = RBP_K562_KD_matches_BAT.filter(pl.col("index").str.contains("CTRL"))

        # Get only unique locations to get the binding pattern for that location

        unique_ctrl_rows = (
            CTRL_rows.with_columns(
                pl.col("index")
                .str.split(by=DELIMITER)
                .list.slice(0, N_ELEMENTS)  # Slice the first N elements of the resulting list
                .alias("prefix")
            )
            .unique(subset="prefix")
            .drop("prefix")
        )

        print("Shape of unique ctrl rows")
        print(unique_ctrl_rows.shape)

        duplicated = pl.concat([
            unique_ctrl_rows.with_columns(pl.lit("CTRL").alias("Row Type")),
            unique_ctrl_rows.with_columns(pl.lit("IS-KD").alias("Row Type"))
        ])

        print("Shape of duplicated:")
        print(duplicated.shape)

        # Do the in-silico KD
        
        cols = [f"{rbp}_{i}_binding" for i in range(1, 7)]
    
        IS_KD_df = duplicated.with_columns([
            pl.when(pl.col("Row Type") == "IS-KD")
            .then(0)
            .otherwise(pl.col(c))
            .alias(c)
            for c in cols
        ])

        # Keep only binding cols, index, and row type

        final = IS_KD_df.select(
            "index",
            "Row Type",
            cs.ends_with("_binding")
        )

        print(f"The shape of final df for {rbp} in K562 is {final.shape}")
        
        return final

In [11]:
test = load_rmats_K562(rbps, BAT_K562)

The shape of the rMATS file for IGF2BP1 in K562 is (55217, 15)
Here is the shape of matches between IGF2BP1 KD events and the BAT for K562:
(4424090, 866)
Shape of unique ctrl rows
(31467, 866)
Shape of duplicated:
(62934, 867)
The shape of final df for IGF2BP1 in K562 is (62934, 836)


In [31]:
test.head()

index,Row Type,AARS_1_binding,AATF_1_binding,ABCF1_1_binding,ADAT1_1_binding,AGGF1_1_binding,AKAP1_1_binding,AKAP8L_1_binding,APEX1_1_binding,APOBEC3C_1_binding,AQR_1_binding,BUD13_1_binding,CPEB4_1_binding,CPSF6_1_binding,CSTF2T_1_binding,DDX1_1_binding,DDX21_1_binding,DDX24_1_binding,DDX3X_1_binding,DDX42_1_binding,DDX43_1_binding,DDX47_1_binding,DDX51_1_binding,DDX52_1_binding,DDX55_1_binding,DDX6_1_binding,DGCR8_1_binding,DHX30_1_binding,DROSHA_1_binding,EEF2_1_binding,EFTUD2_1_binding,EIF3G_1_binding,EIF4E_1_binding,EIF4G2_1_binding,ELAC2_1_binding,ELAVL1_1_binding,EWSR1_1_binding,EXOSC10_1_binding,EXOSC5_1_binding,FAM120A_1_binding,FASTKD2_1_binding,FMR1_1_binding,FTO_1_binding,FUS_1_binding,FXR1_1_binding,FXR2_1_binding,GARS_1_binding,GEMIN5_1_binding,GNL3_1_binding,GPKOW_1_binding,GRWD1_1_binding,GTF2F1_1_binding,HLTF_1_binding,HNRNPA1_1_binding,HNRNPC_1_binding,HNRNPK_1_binding,HNRNPL_1_binding,HNRNPM_1_binding,HNRNPU_1_binding,HNRNPUL1_1_binding,IGF2BP1_1_binding,IGF2BP2_1_binding,ILF3_1_binding,KHDRBS1_1_binding,KHSRP_1_binding,LARP4_1_binding,LARP7_1_binding,LIN28B_1_binding,LSM11_1_binding,MATR3_1_binding,MBNL1_1_binding,METAP2_1_binding,METTL1_1_binding,MORC2_1_binding,MTPAP_1_binding,NCBP2_1_binding,NIPBL_1_binding,NOLC1_1_binding,NONO_1_binding,NPM1_1_binding,NSUN2_1_binding,PABPC4_1_binding,PCBP1_1_binding,PHF6_1_binding,PPIL4_1_binding,PRPF8_1_binding,PTBP1_1_binding,PUM1_1_binding,PUM2_1_binding,PUS1_1_binding,QKI_1_binding,RBFOX2_1_binding,RBM15_1_binding,RBM22_1_binding,RNF187_1_binding,RPS10_1_binding,RPS11_1_binding,RPS3_1_binding,RPS6_1_binding,RYBP_1_binding,SAFB_1_binding,SAFB2_1_binding,SBDS_1_binding,SDAD1_1_binding,SERBP1_1_binding,SF3B1_1_binding,SF3B4_1_binding,SLBP_1_binding,SLTM_1_binding,SMNDC1_1_binding,SND1_1_binding,SRSF1_1_binding,SRSF7_1_binding,SRSF9_1_binding,SSB_1_binding,SUPV3L1_1_binding,TAF15_1_binding,TARDBP_1_binding,TBRG4_1_binding,TIA1_1_binding,TRA2A_1_binding,TROVE2_1_binding,U2AF1_1_binding,U2AF2_1_binding,UCHL5_1_binding,UPF1_1_binding,UTP18_1_binding,UTP3_1_binding,WDR3_1_binding,WDR43_1_binding,WRN_1_binding,XRCC6_1_binding,XRN2_1_binding,YBX3_1_binding,YWHAG_1_binding,ZC3H11A_1_binding,ZC3H8_1_binding,ZNF622_1_binding,ZNF800_1_binding,ZRANB2_1_binding,AARS_2_binding,AATF_2_binding,ABCF1_2_binding,ADAT1_2_binding,AGGF1_2_binding,AKAP1_2_binding,AKAP8L_2_binding,APEX1_2_binding,APOBEC3C_2_binding,AQR_2_binding,BUD13_2_binding,CPEB4_2_binding,CPSF6_2_binding,CSTF2T_2_binding,DDX1_2_binding,DDX21_2_binding,DDX24_2_binding,DDX3X_2_binding,DDX42_2_binding,DDX43_2_binding,DDX47_2_binding,DDX51_2_binding,DDX52_2_binding,DDX55_2_binding,DDX6_2_binding,DGCR8_2_binding,DHX30_2_binding,DROSHA_2_binding,EEF2_2_binding,EFTUD2_2_binding,EIF3G_2_binding,EIF4E_2_binding,EIF4G2_2_binding,ELAC2_2_binding,ELAVL1_2_binding,EWSR1_2_binding,EXOSC10_2_binding,EXOSC5_2_binding,FAM120A_2_binding,FASTKD2_2_binding,FMR1_2_binding,FTO_2_binding,FUS_2_binding,FXR1_2_binding,FXR2_2_binding,GARS_2_binding,GEMIN5_2_binding,GNL3_2_binding,GPKOW_2_binding,GRWD1_2_binding,GTF2F1_2_binding,HLTF_2_binding,HNRNPA1_2_binding,HNRNPC_2_binding,HNRNPK_2_binding,HNRNPL_2_binding,HNRNPM_2_binding,HNRNPU_2_binding,HNRNPUL1_2_binding,IGF2BP1_2_binding,IGF2BP2_2_binding,ILF3_2_binding,KHDRBS1_2_binding,KHSRP_2_binding,LARP4_2_binding,LARP7_2_binding,LIN28B_2_binding,LSM11_2_binding,MATR3_2_binding,MBNL1_2_binding,METAP2_2_binding,METTL1_2_binding,MORC2_2_binding,MTPAP_2_binding,NCBP2_2_binding,NIPBL_2_binding,NOLC1_2_binding,NONO_2_binding,NPM1_2_binding,NSUN2_2_binding,PABPC4_2_binding,PCBP1_2_binding,PHF6_2_binding,PPIL4_2_binding,PRPF8_2_binding,PTBP1_2_binding,PUM1_2_binding,PUM2_2_binding,PUS1_2_binding,QKI_2_binding,RBFOX2_2_binding,RBM15_2_binding,RBM22_2_binding,RNF187_2_binding,RPS10_2_binding,RPS11_2_binding,RPS3_2_binding,RPS6_2_binding,RYBP_2_binding,SAFB_2_binding,SAFB2_2_binding,SBDS_2_binding,SDAD1_2_binding,SERBP1_2_binding,SF3B1_2_binding,SF3B4_2_binding,SLBP_2_binding,SLTM_2_binding,SMNDC1_2_binding,S

In [22]:
print(type(test))

<class 'polars.dataframe.frame.DataFrame'>


## Generating SHAP

In [16]:
import sys, os, argparse, glob, gzip, pickle, shap, gc
import pandas as pd, polars as pl, numpy as np
from loguru import logger

MODEL_DIR = "/project/PlatigLab/users/yogi/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/04_run_final_models_and_SHAP/outputs/pickled_models/XGBRegressor"

In [46]:
K562_hashes = ["kzbv", "mzdv", "niwy", "nuyh", "oodr"]
HepG2_hashes = ["ajtg", "btjy", "gufp", "jmhq", "juwg"]

In [50]:
def main(hashes, binding_pattern):
    """
    Run TreeSHAP for each hash and return average SHAP values
    """

    binding_cols = [col for col in binding_pattern.collect_schema().names() if col.endswith("_binding")]

    # Select all data for SHAP analysis (same for all hashes)
    all_data = (
        binding_pattern
        .select(binding_cols)
        .to_pandas()
    )

    shap_stack = []

    for hash in hashes:

        print(hash)

        # Load model
        with gzip.open(f"{MODEL_DIR}/{hash}.pkl.gz", 'rb') as f:
            model = pickle.load(f)

        explainer = shap.TreeExplainer(
            model,
            model_output="raw",
            feature_perturbation="tree_path_dependent",
        )

        assert all_data.columns.tolist() == list(model.column_order_when_fitting)

        logger.info(f"{hash} → Retrieving SHAP values for data with shape: {all_data.shape}")

        shap_values = explainer.shap_values(
            all_data,
            approximate=False,
            check_additivity=True,
        )

        shap_stack.append(shap_values)


    # stack: (n_hash, n_rows, n_features)
    shap_stack = np.stack(shap_stack, axis=0)

    # average across hashes
    shap_mean = np.mean(shap_stack, axis=0)


    # convert averaged SHAP → polars
    shap_df = pl.DataFrame(
        shap_mean,
        schema=[col.replace("_binding", "_shap") for col in binding_cols]
    )


    # combine binding + shap
    result = pl.concat([binding_pattern, shap_df], how="horizontal")

    return result

In [51]:
shap_test2 = main(K562_hashes, test)

kzbv


2026-03-30 16:22:12.850 | INFO     | __main__:main:33 - kzbv → Retrieving SHAP values for data with shape: (62934, 834)


mzdv


2026-03-30 16:23:20.120 | INFO     | __main__:main:33 - mzdv → Retrieving SHAP values for data with shape: (62934, 834)


niwy


2026-03-30 16:24:20.728 | INFO     | __main__:main:33 - niwy → Retrieving SHAP values for data with shape: (62934, 834)


nuyh


2026-03-30 16:25:33.173 | INFO     | __main__:main:33 - nuyh → Retrieving SHAP values for data with shape: (62934, 834)


oodr


2026-03-30 16:26:43.293 | INFO     | __main__:main:33 - oodr → Retrieving SHAP values for data with shape: (62934, 834)


In [52]:
shap_test2.head()

index,Row Type,AARS_1_binding,AATF_1_binding,ABCF1_1_binding,ADAT1_1_binding,AGGF1_1_binding,AKAP1_1_binding,AKAP8L_1_binding,APEX1_1_binding,APOBEC3C_1_binding,AQR_1_binding,BUD13_1_binding,CPEB4_1_binding,CPSF6_1_binding,CSTF2T_1_binding,DDX1_1_binding,DDX21_1_binding,DDX24_1_binding,DDX3X_1_binding,DDX42_1_binding,DDX43_1_binding,DDX47_1_binding,DDX51_1_binding,DDX52_1_binding,DDX55_1_binding,DDX6_1_binding,DGCR8_1_binding,DHX30_1_binding,DROSHA_1_binding,EEF2_1_binding,EFTUD2_1_binding,EIF3G_1_binding,EIF4E_1_binding,EIF4G2_1_binding,ELAC2_1_binding,ELAVL1_1_binding,EWSR1_1_binding,EXOSC10_1_binding,EXOSC5_1_binding,FAM120A_1_binding,FASTKD2_1_binding,FMR1_1_binding,FTO_1_binding,FUS_1_binding,FXR1_1_binding,FXR2_1_binding,GARS_1_binding,GEMIN5_1_binding,GNL3_1_binding,GPKOW_1_binding,GRWD1_1_binding,GTF2F1_1_binding,HLTF_1_binding,HNRNPA1_1_binding,HNRNPC_1_binding,HNRNPK_1_binding,HNRNPL_1_binding,HNRNPM_1_binding,HNRNPU_1_binding,HNRNPUL1_1_binding,IGF2BP1_1_binding,IGF2BP2_1_binding,ILF3_1_binding,KHDRBS1_1_binding,KHSRP_1_binding,LARP4_1_binding,LARP7_1_binding,LIN28B_1_binding,LSM11_1_binding,MATR3_1_binding,MBNL1_1_binding,METAP2_1_binding,METTL1_1_binding,MORC2_1_binding,MTPAP_1_binding,NCBP2_1_binding,NIPBL_1_binding,NOLC1_1_binding,NONO_1_binding,NPM1_1_binding,NSUN2_1_binding,PABPC4_1_binding,PCBP1_1_binding,PHF6_1_binding,PPIL4_1_binding,PRPF8_1_binding,PTBP1_1_binding,PUM1_1_binding,PUM2_1_binding,PUS1_1_binding,QKI_1_binding,RBFOX2_1_binding,RBM15_1_binding,RBM22_1_binding,RNF187_1_binding,RPS10_1_binding,RPS11_1_binding,RPS3_1_binding,RPS6_1_binding,RYBP_1_binding,SAFB_1_binding,SAFB2_1_binding,SBDS_1_binding,SDAD1_1_binding,SERBP1_1_binding,SF3B1_1_binding,SF3B4_1_binding,SLBP_1_binding,SLTM_1_binding,SMNDC1_1_binding,SND1_1_binding,SRSF1_1_binding,SRSF7_1_binding,SRSF9_1_binding,SSB_1_binding,SUPV3L1_1_binding,TAF15_1_binding,TARDBP_1_binding,TBRG4_1_binding,TIA1_1_binding,TRA2A_1_binding,TROVE2_1_binding,U2AF1_1_binding,U2AF2_1_binding,UCHL5_1_binding,UPF1_1_binding,UTP18_1_binding,UTP3_1_binding,WDR3_1_binding,WDR43_1_binding,WRN_1_binding,XRCC6_1_binding,XRN2_1_binding,YBX3_1_binding,YWHAG_1_binding,ZC3H11A_1_binding,ZC3H8_1_binding,ZNF622_1_binding,ZNF800_1_binding,ZRANB2_1_binding,AARS_2_binding,AATF_2_binding,ABCF1_2_binding,ADAT1_2_binding,AGGF1_2_binding,AKAP1_2_binding,AKAP8L_2_binding,APEX1_2_binding,APOBEC3C_2_binding,AQR_2_binding,BUD13_2_binding,CPEB4_2_binding,CPSF6_2_binding,CSTF2T_2_binding,DDX1_2_binding,DDX21_2_binding,DDX24_2_binding,DDX3X_2_binding,DDX42_2_binding,DDX43_2_binding,DDX47_2_binding,DDX51_2_binding,DDX52_2_binding,DDX55_2_binding,DDX6_2_binding,DGCR8_2_binding,DHX30_2_binding,DROSHA_2_binding,EEF2_2_binding,EFTUD2_2_binding,EIF3G_2_binding,EIF4E_2_binding,EIF4G2_2_binding,ELAC2_2_binding,ELAVL1_2_binding,EWSR1_2_binding,EXOSC10_2_binding,EXOSC5_2_binding,FAM120A_2_binding,FASTKD2_2_binding,FMR1_2_binding,FTO_2_binding,FUS_2_binding,FXR1_2_binding,FXR2_2_binding,GARS_2_binding,GEMIN5_2_binding,GNL3_2_binding,GPKOW_2_binding,GRWD1_2_binding,GTF2F1_2_binding,HLTF_2_binding,HNRNPA1_2_binding,HNRNPC_2_binding,HNRNPK_2_binding,HNRNPL_2_binding,HNRNPM_2_binding,HNRNPU_2_binding,HNRNPUL1_2_binding,IGF2BP1_2_binding,IGF2BP2_2_binding,ILF3_2_binding,KHDRBS1_2_binding,KHSRP_2_binding,LARP4_2_binding,LARP7_2_binding,LIN28B_2_binding,LSM11_2_binding,MATR3_2_binding,MBNL1_2_binding,METAP2_2_binding,METTL1_2_binding,MORC2_2_binding,MTPAP_2_binding,NCBP2_2_binding,NIPBL_2_binding,NOLC1_2_binding,NONO_2_binding,NPM1_2_binding,NSUN2_2_binding,PABPC4_2_binding,PCBP1_2_binding,PHF6_2_binding,PPIL4_2_binding,PRPF8_2_binding,PTBP1_2_binding,PUM1_2_binding,PUM2_2_binding,PUS1_2_binding,QKI_2_binding,RBFOX2_2_binding,RBM15_2_binding,RBM22_2_binding,RNF187_2_binding,RPS10_2_binding,RPS11_2_binding,RPS3_2_binding,RPS6_2_binding,RYBP_2_binding,SAFB_2_binding,SAFB2_2_binding,SBDS_2_binding,SDAD1_2_binding,SERBP1_2_binding,SF3B1_2_binding,SF3B4_2_binding,SLBP_2_binding,SLTM_2_binding,SMNDC1_2_binding,S

## QC checks

In [ ]:
# Get events from the BAT that match SAFB KD events and return those with SAFB binding at pos 3

In [ ]:
SAFB_sig = SAFB_K562_KD_matches_BAT.filter(pl.col("SAFB_3_binding") == 1)

In [ ]:
print(SAFB_sig.shape)

In [ ]:
# Filter for rows where SAFB is bound
binding_cols = ["SAFB_1_binding", "SAFB_2_binding", "SAFB_3_binding", 
                "SAFB_4_binding", "SAFB_5_binding", "SAFB_6_binding"]

SAFB_bound_K562 = SAFB_K562_KD_matches_BAT.filter(
    pl.any_horizontal([pl.col(c) == 1 for c in binding_cols])
)

In [ ]:
print(SAFB_bound_K562.shape)

In [ ]:
df = SAFB_bound_K562.drop([col for col in SAFB_bound_K562.columns if 'shap' in col])
df.write_csv("check_event.csv")

In [ ]:
# Filter for rows where SAFB is bound
binding_cols = ["SAFB_1_binding", "SAFB_2_binding", "SAFB_3_binding", 
                "SAFB_4_binding", "SAFB_5_binding", "SAFB_6_binding"]

for col in binding_cols: 
    print(SAFB_K562_KD_matches_BAT.filter(
        pl.col(col) ==1
    ).unique(
          subset=["Upstream Exon", "Downstream Exon", "Main Exon"]
    ).height)

In [ ]:
for col in binding_cols:
    result = SAFB_K562_KD_matches_BAT.filter(
        pl.col(col) == 1
    ).unique(
        subset=["Upstream Exon", "Downstream Exon", "Main Exon"]
    )
    print(f"\n{col}: {result.height} rows")
    print(result.select(["Upstream Exon", "Downstream Exon", "Main Exon"]))

In [ ]:
SF3B4_check = SAFB_nonsig.filter(
    (pl.col('RBP_KD_Target') == 'SF3B4') & (pl.col('rMATS Event ID') == 56122)
)

In [ ]:
SF3B4_check